In [14]:
import numpy as np
import pandas as pd
import requests
import io

In [15]:
# 자기 상대 파일 경로로 변경해도 됨
df = pd.read_excel('data\서울특별시 모기예보제 정보.xlsx')
df.head()

,mosquito_date,mosquito_value_house,mosquito_value_park
0,2016-05-01,254.4,254.4
1,2016-05-02,273.5,273.5
2,2016-05-03,304.0,304.0
3,2016-05-04,256.2,256.2
4,2016-05-05,243.8,243.8


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mosquito_date         1849 non-null   object 
 1   mosquito_value_house  1849 non-null   float64
 2   mosquito_value_park   1849 non-null   float64
dtypes: float64(2), object(1)
memory usage: 43.5+ KB


In [17]:
# 날짜 데이터 타입 변환 및 2020기준 후의 데이터만 사용, 인덱싱 초기화
df['mosquito_date'] = pd.to_datetime(df['mosquito_date'])
df = df[df['mosquito_date'] >= '2020-01-01']
df = df.reset_index(drop=True)
df.head()

,mosquito_date,mosquito_value_house,mosquito_value_park
0,2020-05-01,32.6,47.0
1,2020-05-02,36.6,52.7
2,2020-05-03,43.0,62.0
3,2020-05-04,43.2,62.3
4,2020-05-05,46.7,67.3


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1127 entries, 0 to 1126
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   mosquito_date         1127 non-null   datetime64[ns]
 1   mosquito_value_house  1127 non-null   float64       
 2   mosquito_value_park   1127 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 26.5 KB


In [30]:
def format_date_for_api(df):
    # 1. 안전하게 날짜(datetime) 타입으로 변환
    df['mosquito_date'] = pd.to_datetime(df['mosquito_date'])
    
    # 2. API가 요구하는 'YYYYMMDD0900' 형식의 문자열로 변환하여 새 컬럼 생성
    df['tm'] = df['mosquito_date'].dt.strftime('%Y%m%d')
    
    return df

def fetch_seoul_asos(df, auth_key):
    df = format_date_for_api(df)
    raw_data_list = []
    
    # 중복 호출을 막기 위해 고유한 날짜(tm)만 반복
    for tm in df['tm'].unique():
        # stn=108 (서울), help=0 (불필요한 설명줄 제거)
        # print(tm)
        url = f"https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd.php?tm={tm}&stn=108&help=0&authKey={auth_key}"
        
        response = requests.get(url)
        
        # response 상태 확인용 print 출력문
        # print(response.status_code)
        
        # 응답이 정상(200)일 때만 텍스트 데이터 저장
        if response.status_code == 200:
            raw_data_list.append(response.text)
            
    return raw_data_list

def parse_kma_text(response_text):
    weather_columns = [
        'TM', 'STN', 'WS_AVG', 'WR_DAY', 'WD_MAX', 'WS_MAX', 'WS_MAX_TM', 'WD_INS', 'WS_INS', 'WS_INS_TM',
        'TA_AVG', 'TA_MAX', 'TA_MAX_TM', 'TA_MIN', 'TA_MIN_TM', 'TD_AVG', 'TS_AVG', 'TG_MIN', 'HM_AVG', 'HM_MIN',
        'HM_MIN_TM', 'PV_AVG', 'EV_S', 'EV_L', 'FG_DUR', 'PA_AVG', 'PS_AVG', 'PS_MAX', 'PS_MAX_TM', 'PS_MIN',
        'PS_MIN_TM', 'CA_TOT', 'SS_DAY', 'SS_DUR', 'SS_CMB', 'SI_DAY', 'SI_60M_MAX', 'SI_60M_MAX_TM', 'RN_DAY', 'RN_D99',
        'RN_DUR', 'RN_60M_MAX', 'RN_60M_MAX_TM', 'RN_10M_MAX', 'RN_10M_MAX_TM', 'RN_POW_MAX', 'RN_POW_MAX_TM', 'SD_NEW', 'SD_NEW_TM', 'SD_MAX',
        'SD_MAX_TM', 'TE_05', 'TE_10', 'TE_15', 'TE_30', 'TE_50'
    ]
    try:
        # 1. 핵심 변경 포인트: sep=',' (콤마 구분자로 변경)
        df_kma = pd.read_csv(io.StringIO(response_text), sep=',', comment='#', header=None)
        
        # (선택) 데이터 끝에 콤마가 있어서 생기는 의미 없는 빈 열(NaN) 제거
        df_kma = df_kma.dropna(axis=1, how='all')
        
    except pd.errors.EmptyDataError:
        return pd.DataFrame() # 데이터가 없는 날은 빈 데이터프레임 반환

    
    # 2. 필요한 데이터 열(인덱스) 추출
    # 이미지의 콤마 순서를 세어보면 대략 0:날짜, 1:지점, 10:평균기온, 17:평균습도, 37:일강수량 입니다.
    # (실제 컬럼 위치가 맞는지 한 번 더 확인해 주세요)
    # df_extracted = df_kma[[0, 1, 10, 17, 37]]
    df_kma.columns = weather_columns
    
    # data frame 확인용 print 출력문
    # display(df_kma)
    
    return df_kma

In [31]:
# 본인 인증키로 변경 부탁드립니다
auth_key = 'l_38MrcRRpm9_DK3EbaZwg'

weather_list = fetch_seoul_asos(df.head(), auth_key)

mos_data_list = []

for item in weather_list:
    parse_kma_text(item)

C:\Users\chlal\AppData\Local\Temp\ipykernel_26256\2265465518.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mosquito_date'] = pd.to_datetime(df['mosquito_date'])
C:\Users\chlal\AppData\Local\Temp\ipykernel_26256\2265465518.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tm'] = df['mosquito_date'].dt.strftime('%Y%m%d')


In [32]:
from tqdm.auto import tqdm  # 로딩바 라이브러리

weather_columns = [
        'TM', 'STN', 'WS_AVG', 'WR_DAY', 'WD_MAX', 'WS_MAX', 'WS_MAX_TM', 'WD_INS', 'WS_INS', 'WS_INS_TM',
        'TA_AVG', 'TA_MAX', 'TA_MAX_TM', 'TA_MIN', 'TA_MIN_TM', 'TD_AVG', 'TS_AVG', 'TG_MIN', 'HM_AVG', 'HM_MIN',
        'HM_MIN_TM', 'PV_AVG', 'EV_S', 'EV_L', 'FG_DUR', 'PA_AVG', 'PS_AVG', 'PS_MAX', 'PS_MAX_TM', 'PS_MIN',
        'PS_MIN_TM', 'CA_TOT', 'SS_DAY', 'SS_DUR', 'SS_CMB', 'SI_DAY', 'SI_60M_MAX', 'SI_60M_MAX_TM', 'RN_DAY', 'RN_D99',
        'RN_DUR', 'RN_60M_MAX', 'RN_60M_MAX_TM', 'RN_10M_MAX', 'RN_10M_MAX_TM', 'RN_POW_MAX', 'RN_POW_MAX_TM', 'SD_NEW', 'SD_NEW_TM', 'SD_MAX',
        'SD_MAX_TM', 'TE_05', 'TE_10', 'TE_15', 'TE_30', 'TE_50'
    ]

# 1. 데이터 수집
auth_key = 'l_38MrcRRpm9_DK3EbaZwg'
# 실제 전체 데이터를 가져오려면 df.head() 대신 df를 사용하세요.
weather_list = fetch_seoul_asos(df, auth_key) 

mos_w_data_list = []

# 2. 로딩바(tqdm)를 적용하여 파싱 진행
for item in tqdm(weather_list, desc="기상 데이터 파싱 중"):
    # parse_kma_text가 리스트 형태를 반환한다고 가정합니다.
    mos_w_data_list.append(parse_kma_text(item))

# 3. 데이터프레임 생성 (이전의 56개 컬럼명 리스트 사용)
weather_df = pd.concat(mos_w_data_list, ignore_index=True)

# 4. 데이터 정보 확인
print("\n--- 데이터프레임 정보 ---")
weather_df.info()

# 5. CSV 파일로 저장
# 한글 깨짐 방지를 위해 encoding='utf-8-sig'를 추천하지만, 요청하신 대로 utf-8로 설정합니다.
weather_df.to_csv('seoul_weather_data.csv', index=False, encoding='utf-8')

print("\n저장이 완료되었습니다.")

기상 데이터 파싱 중:   0%|          | 0/1104 [00:00<?, ?it/s]


--- 데이터프레임 정보 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 56 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TM             1104 non-null   int64  
 1   STN            1104 non-null   int64  
 2   WS_AVG         1104 non-null   float64
 3   WR_DAY         1104 non-null   int64  
 4   WD_MAX         1104 non-null   int64  
 5   WS_MAX         1104 non-null   float64
 6   WS_MAX_TM      1104 non-null   int64  
 7   WD_INS         1104 non-null   int64  
 8   WS_INS         1104 non-null   float64
 9   WS_INS_TM      1104 non-null   int64  
 10  TA_AVG         1104 non-null   float64
 11  TA_MAX         1104 non-null   float64
 12  TA_MAX_TM      1104 non-null   int64  
 13  TA_MIN         1104 non-null   float64
 14  TA_MIN_TM      1104 non-null   int64  
 15  TD_AVG         1104 non-null   float64
 16  TS_AVG         1104 non-null   float64
 17  TG_MIN         1104 non-null   fl

In [35]:
weather_df = pd.read_csv('seoul_weather_data.csv')

In [36]:
# 1. 날짜 형식 통일
# 기상 데이터(weather_df)의 'TM' 컬럼을 모기 데이터의 'tm' 형식(예: '20250101')으로 변환합니다.
mosquito_df = format_date_for_api(df)
weather_df['TM'] = pd.to_datetime(weather_df['TM'].astype(str), format='%Y%m%d').dt.strftime('%Y%m%d')

# 2. 모기 데이터(mosquito_df)를 왼쪽에 두고 병합
# mosquito_df의 날짜 컬럼은 'tm', weather_df는 'TM'이라고 가정합니다.
merged_df = pd.merge(mosquito_df, weather_df, left_on='tm', right_on='TM', how='inner')

# 3. 중복되는 날짜 컬럼(TM) 제거 및 결과 확인
merged_df = merged_df.drop(columns=['TM'])

print("--- [merged_df] 병합 완료 ---")
print(f"전체 컬럼 수: {len(merged_df.columns)}개")
display(merged_df.head())

--- [merged_df] 병합 완료 ---
전체 컬럼 수: 59개


,mosquito_date,mosquito_value_house,mosquito_value_park,tm,STN,WS_AVG,WR_DAY,WD_MAX,WS_MAX,WS_MAX_TM,...,RN_POW_MAX_TM,SD_NEW,SD_NEW_TM,SD_MAX,SD_MAX_TM,TE_05,TE_10,TE_15,TE_30,TE_50
0,2020-05-01,32.6,47.0,20200501,108,2.7,2328,29,5.5,1726,...,-9,-9.0,-9,-9.0,-9,14.9,13.1,12.6,12.8,13.4
1,2020-05-02,36.6,52.7,20200502,108,2.3,1985,23,4.6,111,...,-9,-9.0,-9,-9.0,-9,15.8,13.4,12.7,12.8,13.4
2,2020-05-03,43.0,62.0,20200503,108,2.0,1705,16,3.9,1242,...,-9,-9.0,-9,-9.0,-9,16.2,13.7,12.9,12.9,13.4
3,2020-05-04,43.2,62.3,20200504,108,2.7,2310,27,4.6,1247,...,-9,-9.0,-9,-9.0,-9,16.9,14.1,13.0,12.9,13.4
4,2020-05-05,46.7,67.3,20200505,108,1.9,1601,25,5.3,1354,...,-9,-9.0,-9,-9.0,-9,17.4,14.4,13.2,12.9,13.4


In [37]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1127 entries, 0 to 1126
Data columns (total 59 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   mosquito_date         1127 non-null   datetime64[ns]
 1   mosquito_value_house  1127 non-null   float64       
 2   mosquito_value_park   1127 non-null   float64       
 3   tm                    1127 non-null   object        
 4   STN                   1127 non-null   int64         
 5   WS_AVG                1127 non-null   float64       
 6   WR_DAY                1127 non-null   int64         
 7   WD_MAX                1127 non-null   int64         
 8   WS_MAX                1127 non-null   float64       
 9   WS_MAX_TM             1127 non-null   int64         
 10  WD_INS                1127 non-null   int64         
 11  WS_INS                1127 non-null   float64       
 12  WS_INS_TM             1127 non-null   int64         
 13  TA_AVG            

In [38]:
# 모기 데이터 날짜 중복 확인 (결과가 1보다 크면 중복)
print("모기 데이터 날짜 중복 상위:\n", mosquito_df['tm'].value_counts().head())

# 기상 데이터 날짜 중복 확인
print("\n기상 데이터 날짜 중복 상위:\n", weather_df['TM'].value_counts().head())

모기 데이터 날짜 중복 상위:
 tm
20211006    3
20210724    2
20210904    2
20210707    2
20210915    2
Name: count, dtype: int64

기상 데이터 날짜 중복 상위:
 TM
20200501    1
20231030    1
20240505    1
20240504    1
20240503    1
Name: count, dtype: int64


In [39]:
# 1. 모기 데이터 중복 제거 (날짜별 평균값으로 압축)
# 날짜(tm)를 기준으로 묶어 하루에 하나의 행만 남깁니다.
mosquito_df_unique = mosquito_df.groupby('tm').mean(numeric_only=True).reset_index()

# 2. 기상 데이터 형식 수정 (이미 되어있다면 생략 가능)
weather_df['TM_str'] = pd.to_datetime(weather_df['TM'].astype(str), format='%Y%m%d').dt.strftime('%Y%m%d')

# 3. 깨끗해진 데이터로 재병합
# mosquito_df_unique를 왼쪽에 두어 컬럼 순서를 유지합니다.
merged_df = pd.merge(mosquito_df_unique, weather_df, left_on='tm', right_on='TM_str', how='inner')

# 4. 불필요한 날짜 컬럼(TM, TM_str) 정리
merged_df = merged_df.drop(columns=['TM', 'TM_str'])

print(f"✅ 중복 제거 후 최종 행 개수: {len(merged_df)}개")
display(merged_df.head())

✅ 중복 제거 후 최종 행 개수: 1104개


,tm,mosquito_value_house,mosquito_value_park,STN,WS_AVG,WR_DAY,WD_MAX,WS_MAX,WS_MAX_TM,WD_INS,...,RN_POW_MAX_TM,SD_NEW,SD_NEW_TM,SD_MAX,SD_MAX_TM,TE_05,TE_10,TE_15,TE_30,TE_50
0,20200501,32.6,47.0,108,2.7,2328,29,5.5,1726,29,...,-9,-9.0,-9,-9.0,-9,14.9,13.1,12.6,12.8,13.4
1,20200502,36.6,52.7,108,2.3,1985,23,4.6,111,29,...,-9,-9.0,-9,-9.0,-9,15.8,13.4,12.7,12.8,13.4
2,20200503,43.0,62.0,108,2.0,1705,16,3.9,1242,14,...,-9,-9.0,-9,-9.0,-9,16.2,13.7,12.9,12.9,13.4
3,20200504,43.2,62.3,108,2.7,2310,27,4.6,1247,27,...,-9,-9.0,-9,-9.0,-9,16.9,14.1,13.0,12.9,13.4
4,20200505,46.7,67.3,108,1.9,1601,25,5.3,1354,23,...,-9,-9.0,-9,-9.0,-9,17.4,14.4,13.2,12.9,13.4


In [40]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 58 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tm                    1104 non-null   object 
 1   mosquito_value_house  1104 non-null   float64
 2   mosquito_value_park   1104 non-null   float64
 3   STN                   1104 non-null   int64  
 4   WS_AVG                1104 non-null   float64
 5   WR_DAY                1104 non-null   int64  
 6   WD_MAX                1104 non-null   int64  
 7   WS_MAX                1104 non-null   float64
 8   WS_MAX_TM             1104 non-null   int64  
 9   WD_INS                1104 non-null   int64  
 10  WS_INS                1104 non-null   float64
 11  WS_INS_TM             1104 non-null   int64  
 12  TA_AVG                1104 non-null   float64
 13  TA_MAX                1104 non-null   float64
 14  TA_MAX_TM             1104 non-null   int64  
 15  TA_MIN               

In [41]:
merged_df.to_csv('data\Merge_mos_w_data.csv', index=False, encoding='utf-8')

In [13]:
import requests

s_auth_key = '46646b497963686f373344446a7363'

url = 'http://openapi.seoul.go.kr:8088/{s_auth_key}/xml/WPOSInformationTime/1/5/'

response = requests.get(url)
print(response.content)

b'<RESULT><CODE>INFO-100</CODE><MESSAGE><![CDATA[\xec\x9d\xb8\xec\xa6\x9d\xed\x82\xa4\xea\xb0\x80 \xec\x9c\xa0\xed\x9a\xa8\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.\n\xec\x9d\xb8\xec\xa6\x9d\xed\x82\xa4\xea\xb0\x80 \xec\x97\x86\xeb\x8a\x94 \xea\xb2\xbd\xec\x9a\xb0, \xec\x97\xb4\xeb\xa6\xb0 \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 \xea\xb4\x91\xec\x9e\xa5 \xed\x99\x88\xed\x8e\x98\xec\x9d\xb4\xec\xa7\x80\xec\x97\x90\xec\x84\x9c \xec\x9d\xb8\xec\xa6\x9d\xed\x82\xa4\xeb\xa5\xbc \xec\x8b\xa0\xec\xb2\xad\xed\x95\x98\xec\x8b\xad\xec\x8b\x9c\xec\x98\xa4.]]></MESSAGE></RESULT>'
